# Avaliação da Resiliência de Redes Neurais de Grafos a Concept Drift na Detecção de Botnets em IoT
**Autora:** Letícia Alves de Pontes  
**Orientador:** Prof. Me. Paulo Rogério de Mello Cardoso

Este notebook unifica e concentra todo o pipeline experimental do TCC de forma 100% autocontida (sem dependência de módulos `.py` externos), podendo ser executado diretamente no Google Colab ou localmente:

1. **Configuração de Ambiente, GPU e Armazenamento** (Colab, Google Drive ou Local)
2. **Ingestão, Parsing de Logs Zeek e Divisão Temporal Estrita** (IoT-23 e TON-IoT)
3. **Harmonização para o Esquema Canônico de 14 Colunas** (IoT-23, TON-IoT, CTU-13)
4. **Construção de Grafos, Heurística de Pesos $w(u,v)$ e Snapshots Temporais** (PyTorch Geometric)
5. **Testes Internos de Integridade do Pipeline** (Validação automática com dados sintéticos)
6. **Execução do Pipeline Experimental (Cenário T — IoT-23)**

## 1. Instalação de Dependências, Configuração de Ambiente e Sementes
Detecta automaticamente se está rodando no Google Colab (com ou sem Google Drive montado) ou localmente, instala os pacotes necessários (`torch_geometric`, `pyarrow`, `scikit-learn`, `kaggle`) e fixa todas as sementes aleatórias para reprodutibilidade determinística (`SEED = 42`).

In [6]:
# Instala dependências essenciais no runtime (execução silenciosa)
%pip install -q torch-geometric pyarrow scikit-learn kaggle tqdm matplotlib seaborn

import os
import re
import sys
import random
import tarfile
import logging
import tempfile
import urllib.request
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Generator

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import torch

# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("tcc_pipeline")

SEED: int = int(os.environ.get("EXPERIMENT_SEED", 42))

def set_seed(seed: int = SEED) -> None:
    """Fixa as sementes em bibliotecas padrão, NumPy e PyTorch."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def is_colab() -> bool:
    """Detecta se o código está sendo executado no Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def is_drive_mounted() -> bool:
    """Verifica se o Google Drive está montado no Colab."""
    return Path("/content/drive/MyDrive").exists()

def get_base_dirs() -> Dict[str, Path]:
    """Determina os diretórios de dados e resultados de forma adaptativa."""
    env_data = os.environ.get("DATA_DIR")
    env_results = os.environ.get("RESULTS_DIR")

    if env_data and env_results:
        data_dir = Path(env_data)
        results_dir = Path(env_results)
        logger.info(f"Usando caminhos via variáveis de ambiente: {data_dir}")
    elif is_colab():
        if is_drive_mounted():
            base_drive = Path("/content/drive/MyDrive/tcc")
            data_dir = base_drive / "data"
            results_dir = base_drive / "results"
            logger.info("Google Colab com Google Drive montado detectado.")
        else:
            base_ephemeral = Path("/content/tcc")
            data_dir = base_ephemeral / "data"
            results_dir = base_ephemeral / "results"
            logger.warning(
                "Google Drive NÃO montado no Colab! Usando armazenamento volátil (/content/tcc)."
            )
    else:
        cwd = Path.cwd().resolve()
        repo_root = cwd.parent if cwd.name == "notebooks" else cwd
        data_dir = repo_root / "data"
        results_dir = repo_root / "results"
        logger.info(f"Ambiente local detectado. Diretório base: {repo_root}")

    data_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)
    (data_dir / "parquet").mkdir(parents=True, exist_ok=True)

    return {
        "DATA_DIR": data_dir,
        "PARQUET_DIR": data_dir / "parquet",
        "RESULTS_DIR": results_dir,
    }

set_seed(SEED)
_DIRS = get_base_dirs()
DATA_DIR: Path = _DIRS["DATA_DIR"]
PARQUET_DIR: Path = _DIRS["PARQUET_DIR"]
RESULTS_DIR: Path = _DIRS["RESULTS_DIR"]

print(f"✓ PyTorch: {torch.__version__} | CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Ativa: {torch.cuda.get_device_name(0)}")
print(f"✓ Semente fixada em: {SEED}")
print(f"✓ DATA_DIR:    {DATA_DIR}")
print(f"✓ PARQUET_DIR: {PARQUET_DIR}")
print(f"✓ RESULTS_DIR: {RESULTS_DIR}")

✓ PyTorch: 2.11.0+cu130 | CUDA Disponível: True
✓ GPU Ativa: Tesla T4
✓ Semente fixada em: 42
✓ DATA_DIR:    /content/tcc/data
✓ PARQUET_DIR: /content/tcc/data/parquet
✓ RESULTS_DIR: /content/tcc/results


## 2. Funções de Ingestão, Parsing Zeek, Parquet e Divisão Temporal
Define o pipeline de download do IoT-23 (`conn.log.labeled`) e TON-IoT (via Kaggle), leitura em lotes (*streaming chunks* de baixo consumo de RAM), conversão para formato colunar Parquet (Snappy), partição temporal estrita por `min(ts)` e *undersampling* estratificado exclusivo de treino.

In [7]:
IOT23_LIGHTWEIGHT_URL: str = (
    "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/iot_23_datasets_small.tar.gz"
)
TON_IOT_KAGGLE_DATASET: str = "arnobbhowmik/ton-iot-network-dataset"

ZEEK_DTYPES: Dict[str, str] = {
    "ts": "float64",
    "uid": "string",
    "id.orig_h": "string",
    "id.orig_p": "float32",
    "id.resp_h": "string",
    "id.resp_p": "float32",
    "proto": "category",
    "service": "category",
    "duration": "float32",
    "orig_bytes": "float32",
    "resp_bytes": "float32",
    "conn_state": "category",
    "local_orig": "category",
    "local_resp": "category",
    "missed_bytes": "float32",
    "history": "string",
    "orig_pkts": "float32",
    "orig_ip_bytes": "float32",
    "resp_pkts": "float32",
    "resp_ip_bytes": "float32",
}

def download_iot23_lightweight(
    dest_dir: Path = DATA_DIR,
    url: str = IOT23_LIGHTWEIGHT_URL,
    force_download: bool = False,
) -> Path:
    """Baixa e descompacta a versão leve do dataset IoT-23 contendo conn.log.labeled."""
    dest_dir.mkdir(parents=True, exist_ok=True)
    archive_name = "iot_23_datasets_small.tar.gz"
    archive_path = dest_dir / archive_name
    extracted_flag = dest_dir / ".iot23_extracted"

    if extracted_flag.exists() and not force_download:
        logger.info("Dataset IoT-23 já descompactado em %s", dest_dir)
        return dest_dir

    if not archive_path.exists() or force_download:
        logger.info("Iniciando download do IoT-23 (versão leve) de %s", url)
        with tqdm(unit="B", unit_scale=True, miniters=1, desc=archive_name) as t:
            def reporthook(block_num: int, block_size: int, total_size: int):
                if total_size > 0:
                    t.total = total_size
                t.update(block_size)
            urllib.request.urlretrieve(url, archive_path, reporthook=reporthook)

    logger.info("Descompactando arquivo %s...", archive_path)
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=dest_dir)

    extracted_flag.touch()
    logger.info("Extração concluída com sucesso.")
    return dest_dir

def download_ton_iot_kaggle(
    dest_dir: Optional[Path] = None,
    dataset_slug: str = TON_IOT_KAGGLE_DATASET,
    force_download: bool = False,
) -> Path:
    """Baixa e descompacta o dataset TON-IoT (Network Dataset) a partir do Kaggle."""
    if dest_dir is None:
        dest_dir = DATA_DIR / "ton_iot"
    dest_dir.mkdir(parents=True, exist_ok=True)
    extracted_flag = dest_dir / ".ton_iot_extracted"

    if extracted_flag.exists() and not force_download:
        logger.info("Dataset TON-IoT já extraído em %s", dest_dir)
        return dest_dir

    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files(dataset_slug, path=str(dest_dir), unzip=True, quiet=False)
    extracted_flag.touch()
    logger.info("Download e extração do TON-IoT concluídos em %s", dest_dir)
    return dest_dir

def parse_zeek_header(filepath: Path) -> List[str]:
    """Extrai a lista de colunas da diretiva #fields de um arquivo Zeek conn.log."""
    fields: List[str] = []
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#fields"):
                fields = line.split("\t")[1:]
                break
            if not line.startswith("#"):
                break
    if not fields:
        raise ValueError(f"Não foi possível encontrar '#fields' no arquivo {filepath}")
    return fields

def parse_label_columns(raw_series: pd.Series) -> pd.DataFrame:
    """Efetua o parsing da coluna de rótulo composto do IoT-23 em (label, detailed_label)."""
    def _parse_val(val: str) -> Tuple[str, str]:
        if not isinstance(val, str) or val.strip() == "-" or not val.strip():
            return "Benign", "Benign"
        val_clean = val.strip()
        binary_label = "Benign" if "benign" in val_clean.lower() else "Malicious"
        parts = [p.strip() for p in re.split(r"\s+", val_clean) if p.strip()]
        if len(parts) >= 2:
            detailed = parts[-1]
            if detailed.lower() == "benign":
                detailed = "Benign"
        else:
            detailed = "Unknown" if binary_label == "Malicious" else "Benign"
        return binary_label, detailed

    parsed = raw_series.apply(_parse_val)
    return pd.DataFrame(parsed.tolist(), index=raw_series.index, columns=["label", "detailed_label"])

def read_conn_log_batches(
    filepath: Path,
    chunksize: int = 200_000,
    max_rows: Optional[int] = None,
) -> Generator[pd.DataFrame, None, None]:
    """Lê um arquivo conn.log.labeled em chunks com baixo consumo de memória."""
    raw_fields = parse_zeek_header(filepath)
    rows_yielded = 0

    chunks = pd.read_csv(
        filepath,
        sep="\t",
        comment="#",
        names=raw_fields,
        na_values=["-", "(empty)"],
        dtype=str,
        chunksize=chunksize,
        low_memory=True,
    )

    for chunk in chunks:
        if max_rows and rows_yielded >= max_rows:
            break
        if max_rows and (rows_yielded + len(chunk)) > max_rows:
            chunk = chunk.iloc[: max_rows - rows_yielded].copy()

        last_col = raw_fields[-1]
        labels_df = parse_label_columns(chunk[last_col])
        chunk["label"] = labels_df["label"].astype("category")
        chunk["detailed_label"] = labels_df["detailed_label"].astype("category")

        for col, dtype in ZEEK_DTYPES.items():
            if col in chunk.columns:
                try:
                    if dtype == "category":
                        chunk[col] = chunk[col].astype("category")
                    elif dtype == "string":
                        chunk[col] = chunk[col].astype("string")
                    else:
                        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")
                except Exception:
                    pass

        rows_yielded += len(chunk)
        yield chunk

def convert_capture_to_parquet(
    capture_log_path: Path,
    output_parquet_path: Path,
    chunksize: int = 200_000,
    max_rows: Optional[int] = None,
) -> None:
    """Converte um conn.log.labeled em múltiplos chunks para um único arquivo Parquet (PyArrow)."""
    import pyarrow as pa
    import pyarrow.parquet as pq

    output_parquet_path.parent.mkdir(parents=True, exist_ok=True)
    writer = None
    try:
        for chunk in read_conn_log_batches(capture_log_path, chunksize=chunksize, max_rows=max_rows):
            for c in chunk.select_dtypes(include=["category"]).columns:
                chunk[c] = chunk[c].astype(str)
            table = pa.Table.from_pandas(chunk, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(output_parquet_path, table.schema, compression="snappy")
            writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()

def list_iot23_captures(base_dir: Path = DATA_DIR) -> pd.DataFrame:
    """Varre o diretório em busca de capturas do IoT-23 e ordena por min(ts)."""
    records = []
    log_files = list(base_dir.rglob("conn.log.labeled"))
    if not log_files:
        log_files = list(base_dir.rglob("conn.log"))

    logger.info("Encontrados %d arquivos de log em %s", len(log_files), base_dir)
    for path in log_files:
        capture_name = path.parent.name
        file_size_mb = path.stat().st_size / (1024 * 1024)
        min_ts = None
        line_count = 0
        try:
            with open(path, "r", encoding="utf-8", errors="replace") as f:
                for line in f:
                    if not line.startswith("#"):
                        line_count += 1
                        if min_ts is None:
                            parts = line.split("\t")
                            try:
                                min_ts = float(parts[0])
                            except (ValueError, IndexError):
                                pass
        except Exception as e:
            logger.warning("Erro ao ler metadados de %s: %s", path, e)

        start_date = pd.to_datetime(min_ts, unit="s") if min_ts else pd.NaT
        records.append({
            "capture_name": capture_name,
            "filepath": str(path),
            "size_mb": round(file_size_mb, 2),
            "total_lines_approx": line_count,
            "first_timestamp_epoch": min_ts,
            "derived_start_date": start_date,
        })

    df = pd.DataFrame(records)
    if not df.empty and "derived_start_date" in df.columns:
        df = df.sort_values(by="derived_start_date", ascending=True).reset_index(drop=True)
    return df

def generate_temporal_splits(
    captures_summary_df: pd.DataFrame,
    train_ratio: float = 0.8,
) -> Tuple[List[str], List[str]]:
    """Gera partição temporal em ordem cronológica estrita (sem embaralhamento)."""
    sorted_df = captures_summary_df.sort_values("derived_start_date").reset_index(drop=True)
    n_total = len(sorted_df)
    n_train = max(1, int(n_total * train_ratio))
    train_captures = sorted_df.iloc[:n_train]["capture_name"].tolist()
    test_captures = sorted_df.iloc[n_train:]["capture_name"].tolist()
    return train_captures, test_captures

def stratified_train_undersampling(
    df_train: pd.DataFrame,
    majority_class: int = 1,
    minority_class: int = 0,
    ratio: float = 1.0,
    random_state: int = 42,
) -> pd.DataFrame:
    """Aplica undersampling aleatório estratificado exclusivamente no conjunto de treino."""
    if "label" not in df_train.columns:
        raise ValueError("O DataFrame precisa conter a coluna 'label'.")
    df_min = df_train[df_train["label"] == minority_class]
    df_maj = df_train[df_train["label"] == majority_class]
    n_minority = len(df_min)
    if n_minority == 0 or len(df_maj) == 0:
        return df_train
    n_majority_sample = int(n_minority * ratio)
    if n_majority_sample >= len(df_maj):
        return df_train
    df_maj_sampled = df_maj.sample(n=n_majority_sample, random_state=random_state)
    df_resampled = pd.concat([df_min, df_maj_sampled], ignore_index=True)
    if "ts" in df_resampled.columns:
        df_resampled = df_resampled.sort_values("ts").reset_index(drop=True)
    else:
        df_resampled = df_resampled.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return df_resampled

print("✓ Funções de ingestão, parsing Zeek e divisão temporal carregadas.")

✓ Funções de ingestão, parsing Zeek e divisão temporal carregadas.


## 3. Harmonização de Esquemas de Rede (14 Colunas Canônicas)
Unifica variáveis de diferentes datasets (**IoT-23**, **TON-IoT** e **CTU-13**) no esquema canônico padronizado de 14 atributos, isolando a representação tabular e de arestas.

In [8]:
CANONICAL_COLUMNS: List[str] = [
    "ts", "src_ip", "src_port", "dst_ip", "dst_port", "proto",
    "duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts",
    "label", "detailed_label", "dataset_source",
]

NUMERICAL_FEATURE_COLS: List[str] = [
    "duration", "orig_bytes", "resp_bytes", "orig_pkts", "resp_pkts",
]

def harmonize_iot23(df: pd.DataFrame) -> pd.DataFrame:
    """Harmoniza um DataFrame do IoT-23 para o esquema canônico de 14 colunas."""
    col_map = {
        "id.orig_h": "src_ip",
        "id.orig_p": "src_port",
        "id.resp_h": "dst_ip",
        "id.resp_p": "dst_port",
    }
    res = df.rename(columns=col_map).copy()
    if "ts" not in res.columns:
        raise ValueError("DataFrame do IoT-23 precisa conter a coluna 'ts'.")

    if "label" in res.columns:
        if res["label"].dtype == "object" or res["label"].dtype.name in ("category", "string"):
            res["label"] = res["label"].apply(
                lambda x: 0 if str(x).strip().lower() == "benign" else 1
            ).astype(np.int64)
    else:
        res["label"] = 0

    if "detailed_label" not in res.columns:
        res["detailed_label"] = "Benign"

    for col in NUMERICAL_FEATURE_COLS:
        if col in res.columns:
            res[col] = pd.to_numeric(res[col], errors="coerce").fillna(0.0).astype(np.float32)
        else:
            res[col] = np.float32(0.0)

    res["dataset_source"] = "iot23"
    for col in ["src_ip", "dst_ip", "proto"]:
        res[col] = res[col].astype(str) if col in res.columns else "unknown"

    return res[CANONICAL_COLUMNS]

def harmonize_ton_iot(df: pd.DataFrame) -> pd.DataFrame:
    """Harmoniza o CSV de tráfego de rede do TON-IoT para o esquema canônico."""
    col_map = {
        "src_bytes": "orig_bytes",
        "dst_bytes": "resp_bytes",
        "src_pkts": "orig_pkts",
        "dst_pkts": "resp_pkts",
        "type": "detailed_label",
    }
    res = df.rename(columns=col_map).copy()
    if "label" in res.columns:
        res["label"] = pd.to_numeric(res["label"], errors="coerce").fillna(0).astype(np.int64)
    elif "detailed_label" in res.columns:
        res["label"] = res["detailed_label"].apply(
            lambda x: 0 if str(x).strip().lower() in ("normal", "benign") else 1
        ).astype(np.int64)
    else:
        res["label"] = 0

    if "detailed_label" not in res.columns:
        res["detailed_label"] = res["label"].apply(lambda x: "Benign" if x == 0 else "Malicious")

    for col in NUMERICAL_FEATURE_COLS:
        if col in res.columns:
            res[col] = pd.to_numeric(res[col], errors="coerce").fillna(0.0).astype(np.float32)
        else:
            res[col] = np.float32(0.0)

    res["ts"] = pd.to_numeric(res["ts"], errors="coerce").fillna(0.0).astype(np.float64) if "ts" in res.columns else 0.0
    res["dataset_source"] = "ton_iot"
    for col in ["src_ip", "dst_ip", "proto"]:
        res[col] = res[col].astype(str) if col in res.columns else "unknown"

    return res[CANONICAL_COLUMNS]

def harmonize_ctu13(df: pd.DataFrame) -> pd.DataFrame:
    """Harmoniza o formato NetFlow (.binetflow) do CTU-13 para o esquema canônico."""
    col_map = {
        "SrcAddr": "src_ip",
        "Sport": "src_port",
        "DstAddr": "dst_ip",
        "Dport": "dst_port",
        "Proto": "proto",
        "Dur": "duration",
        "SrcBytes": "orig_bytes",
        "TotBytes": "total_bytes",
        "TotPkts": "orig_pkts",
        "Label": "raw_label",
    }
    res = df.rename(columns=col_map).copy()
    if "resp_bytes" not in res.columns:
        if "total_bytes" in res.columns and "orig_bytes" in res.columns:
            tot = pd.to_numeric(res["total_bytes"], errors="coerce").fillna(0.0)
            src = pd.to_numeric(res["orig_bytes"], errors="coerce").fillna(0.0)
            res["resp_bytes"] = np.maximum(0.0, tot - src).astype(np.float32)
        else:
            res["resp_bytes"] = np.float32(0.0)
    if "resp_pkts" not in res.columns:
        res["resp_pkts"] = np.float32(0.0)
    if "StartTime" in res.columns:
        res["ts"] = pd.to_datetime(res["StartTime"], errors="coerce").astype("int64") // 10**9
    elif "ts" in res.columns:
        res["ts"] = pd.to_numeric(res["ts"], errors="coerce").fillna(0.0).astype(np.float64)
    else:
        res["ts"] = 0.0
    if "raw_label" in res.columns:
        res["label"] = res["raw_label"].apply(lambda x: 1 if "botnet" in str(x).lower() else 0).astype(np.int64)
        res["detailed_label"] = res["raw_label"].astype(str)
    else:
        res["label"] = 0
        res["detailed_label"] = "Unknown"
    for col in NUMERICAL_FEATURE_COLS:
        res[col] = pd.to_numeric(res[col], errors="coerce").fillna(0.0).astype(np.float32) if col in res.columns else np.float32(0.0)
    res["dataset_source"] = "ctu13"
    for col in ["src_ip", "dst_ip", "proto"]:
        res[col] = res[col].astype(str) if col in res.columns else "unknown"
    return res[CANONICAL_COLUMNS]

print("✓ Funções de harmonização canônica carregadas.")

✓ Funções de harmonização canônica carregadas.


## 4. Construção de Grafos, Pesos de Arestas $w(u,v)$ e Snapshots Temporais
Implementa o cálculo da heurística de pesos normalizados por grau de saída:
$$w(u, v) = \frac{f(u, v)}{\sum_i f(u, i)}$$
além da segmentação em janelas temporais (`create_temporal_snapshots`) e conversão para objetos `Data` do *PyTorch Geometric* (`build_pyg_graph_data`).

In [9]:
def compute_edge_weights(df_flows: pd.DataFrame) -> pd.DataFrame:
    """Calcula pesos normalizados das arestas w(u, v) = f(u, v) / sum_i f(u, i)."""
    if "src_ip" not in df_flows.columns or "dst_ip" not in df_flows.columns:
        raise ValueError("O DataFrame deve conter 'src_ip' e 'dst_ip'.")
    edge_counts = (
        df_flows.groupby(["src_ip", "dst_ip"])
        .size()
        .reset_index(name="flow_count")
    )
    out_degree_sum = edge_counts.groupby("src_ip")["flow_count"].transform("sum")
    edge_counts["weight"] = (edge_counts["flow_count"] / out_degree_sum).astype(np.float32)
    return edge_counts

def create_temporal_snapshots(
    df_flows: pd.DataFrame,
    window_seconds: float = 3600.0,
    stride_seconds: Optional[float] = None,
) -> List[pd.DataFrame]:
    """Segmenta os fluxos ordenados temporalmente em uma sequência de janelas (snapshots)."""
    if "ts" not in df_flows.columns:
        raise ValueError("O DataFrame precisa conter a coluna 'ts'.")
    df_sorted = df_flows.sort_values("ts").reset_index(drop=True)
    if df_sorted.empty:
        return []
    min_ts = df_sorted["ts"].iloc[0]
    max_ts = df_sorted["ts"].iloc[-1]
    stride = stride_seconds if stride_seconds is not None else window_seconds
    snapshots = []
    current_start = min_ts
    while current_start <= max_ts:
        current_end = current_start + window_seconds
        mask = (df_sorted["ts"] >= current_start) & (df_sorted["ts"] < current_end)
        snap = df_sorted.loc[mask].copy()
        if not snap.empty:
            snapshots.append(snap)
        current_start += stride
    return snapshots

def build_pyg_graph_data(
    df_flows: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
    node_id_map: Optional[Dict[str, int]] = None,
):
    """Converte um DataFrame de fluxos harmonizados em um objeto Data do PyTorch Geometric."""
    from torch_geometric.data import Data

    if feature_cols is None:
        feature_cols = NUMERICAL_FEATURE_COLS

    all_ips = pd.concat([df_flows["src_ip"], df_flows["dst_ip"]]).unique()
    if node_id_map is None:
        node_id_map = {ip: idx for idx, ip in enumerate(all_ips)}

    src_indices = df_flows["src_ip"].map(node_id_map).values
    dst_indices = df_flows["dst_ip"].map(node_id_map).values
    edge_index = torch.tensor(np.array([src_indices, dst_indices]), dtype=torch.long)

    weights_df = compute_edge_weights(df_flows)
    merged_weights = df_flows.merge(
        weights_df[["src_ip", "dst_ip", "weight"]],
        on=["src_ip", "dst_ip"],
        how="left",
    )
    edge_weight = torch.tensor(merged_weights["weight"].values, dtype=torch.float32)

    present_cols = [c for c in feature_cols if c in df_flows.columns]
    if present_cols:
        edge_attr = torch.tensor(df_flows[present_cols].values, dtype=torch.float32)
    else:
        edge_attr = edge_weight.unsqueeze(1)

    y = torch.tensor(df_flows["label"].values, dtype=torch.long) if "label" in df_flows.columns else None
    num_nodes = len(node_id_map)
    x = torch.ones((num_nodes, 1), dtype=torch.float32)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
        edge_weight=edge_weight,
        y=y,
        num_nodes=num_nodes,
    )
    return data, node_id_map

print("✓ Funções de grafos, pesos de arestas e snapshots temporais carregadas.")

✓ Funções de grafos, pesos de arestas e snapshots temporais carregadas.


## 5. Validação Rápida de Integridade (Auto-Teste Sintético)
Executa os testes unitários diretamente no notebook com uma amostra sintética de `conn.log.labeled` para garantir que o parser Zeek, a harmonização e a construção de grafos estão funcionando perfeitamente antes de processar os dados reais.

In [ ]:
ZEEK_SYNTHETIC_DATA = (
    "#separator \\x09\n"
    "#set_separator ,\n"
    "#empty_field (empty)\n"
    "#unset_field -\n"
    "#path conn\n"
    "#open 2026-10-06-00-00-00\n"
    "#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\tlocal_orig\tlocal_resp\tmissed_bytes\thistory\torig_pkts\torig_ip_bytes\tresp_pkts\tresp_ip_bytes\ttunnel_parents\tlabel\n"
    "1532165000.000000\tC1\t192.168.1.1\t1000\t192.168.1.2\t80\ttcp\thttp\t1.5\t100\t200\tSF\t-\t-\t0\tShADadFf\t10\t500\t12\t600\t-\t-   benign   -\n"
    "1532165010.000000\tC2\t192.168.1.1\t1001\t192.168.1.3\t443\ttcp\t-\t-\t-\t-\tS0\t-\t-\t0\tS\t1\t40\t0\t0\t-\t-   Malicious   PartOfAHorizontalPortScan\n"
    "1532165020.000000\tC3\t192.168.1.5\t5353\t224.0.0.251\t5353\tudp\tdns\t0.05\t50\t0\tSF\t-\t-\t0\tD\t1\t78\t0\t0\t-\tBenign\n"
    "1532165030.000000\tC4\t192.168.1.1\t1002\t192.168.1.4\t23\ttcp\t-\t0.1\t20\t30\tSF\t-\t-\t0\tShA\t2\t80\t2\t80\t-\t(empty)   Malicious   Mirai\n"
)

with tempfile.TemporaryDirectory() as tmpdir:
    cap_dir = Path(tmpdir) / "CTU-IoT-Malicious-1-1"
    cap_dir.mkdir()
    log_file = cap_dir / "conn.log.labeled"
    log_file.write_text(ZEEK_SYNTHETIC_DATA, encoding="utf-8")

    fields = parse_zeek_header(log_file)
    assert len(fields) == 22 and fields[0] == "ts"

    chunks = list(read_conn_log_batches(log_file, chunksize=2))
    df_syn = pd.concat(chunks, ignore_index=True)
    assert len(df_syn) == 4 and pd.isna(df_syn.loc[1, "duration"])

    df_harm = harmonize_iot23(df_syn)
    assert list(df_harm.columns) == CANONICAL_COLUMNS
    assert df_harm["label"].tolist() == [0, 1, 0, 1]

    weights = compute_edge_weights(df_harm)
    assert np.isclose(weights[weights["src_ip"] == "192.168.1.1"]["weight"].sum(), 1.0)

    pyg_data, node_map = build_pyg_graph_data(df_harm)
    assert pyg_data.num_nodes == 6 and pyg_data.edge_index.shape == (2, 4)

print("✓ Todos os testes de integridade (Zeek Parser, Harmonização, Pesos e PyG Data) passaram com sucesso!")

## 6. Execução do Pipeline de Dados (Cenário T — IoT-23)
As células abaixo executam o download oficial da versão leve do **IoT-23**, listam e ordenam cronologicamente as capturas por `min(ts)`, convertem cada captura para Parquet e geram a divisão temporal estrita de treino e teste.

In [ ]:
TRAIN_RATIO = 0.70  # 70% capturas mais antigas para treino, 30% mais recentes para teste

# 6.1 Download e extração do IoT-23 (pula automaticamente se já extraído)
download_iot23_lightweight(dest_dir=DATA_DIR)

In [ ]:
# 6.2 Varredura dos logs e derivação cronológica a partir de min(ts)
df_captures = list_iot23_captures(base_dir=DATA_DIR)
print(f"Total de capturas identificadas: {len(df_captures)}")
display(df_captures[["capture_name", "size_mb", "total_lines_approx", "derived_start_date"]])

In [ ]:
# 6.3 Conversão de cada captura individualmente para Parquet (baixo uso de RAM)
for idx, row in df_captures.iterrows():
    raw_path = Path(row["filepath"])
    out_parquet = PARQUET_DIR / f"{row['capture_name']}.parquet"
    if not out_parquet.exists():
        print(f"Convertendo [{idx+1}/{len(df_captures)}]: {row['capture_name']}...")
        convert_capture_to_parquet(
            capture_log_path=raw_path,
            output_parquet_path=out_parquet,
            chunksize=200_000,
            max_rows=None,
        )
print("✓ Todas as capturas foram salvas em formato Parquet.")

In [ ]:
# 6.4 Estatísticas e Visualização da Proporção de Tráfego Malicioso por Captura
stats = []
for p_file in sorted(PARQUET_DIR.glob("*.parquet")):
    df_chunk = pd.read_parquet(p_file, columns=["label", "detailed_label", "id.orig_h", "id.resp_h"])
    tot = len(df_chunk)
    n_mal = (df_chunk["label"] == "Malicious").sum()
    stats.append({
        "capture": p_file.stem,
        "total_flows": tot,
        "malicious_pct": (n_mal / tot * 100) if tot > 0 else 0,
        "unique_src_ips": df_chunk["id.orig_h"].nunique(),
        "unique_dst_ips": df_chunk["id.resp_h"].nunique(),
        "attacks": list(df_chunk[df_chunk["label"] == "Malicious"]["detailed_label"].unique()),
    })

df_stats = pd.DataFrame(stats)
if not df_stats.empty:
    display(df_stats.drop(columns=["attacks"]))
    plt.figure(figsize=(12, 4))
    plt.bar(df_stats["capture"], df_stats["malicious_pct"], color="crimson", alpha=0.7)
    plt.xticks(rotation=90)
    plt.ylabel("% Malicioso")
    plt.title("Proporção de Tráfego Malicioso por Captura do IoT-23")
    plt.tight_layout()
    plt.show()

In [ ]:
# 6.5 Geração da Divisão Temporal Estrita (Cenário T)
if not df_captures.empty:
    train_caps, test_caps = generate_temporal_splits(df_captures, train_ratio=TRAIN_RATIO)
    print(f"Capturas de Treino ({len(train_caps)}): {train_caps}")
    print(f"Capturas de Teste ({len(test_caps)}): {test_caps}")

    train_list_md = "\n".join([f"- `{c}`" for c in train_caps])
    test_list_md = "\n".join([f"- `{c}`" for c in test_caps])
    splits_content = (
        "# Divisão Temporal Provisória (Cenário T - IoT-23)\n\n"
        "- **Critério de Divisão:** Ordenação cronológica estrita baseada no primeiro timestamp real (`min(ts)`) extraído do `conn.log.labeled`.\n"
        f"- **Proporção Alvo:** {int(TRAIN_RATIO*100)}% capturas mais antigas para treino, {int((1-TRAIN_RATIO)*100)}% mais recentes para teste.\n\n"
        f"## Capturas de Treino ({len(train_caps)})\n{train_list_md}\n\n"
        f"## Capturas de Teste ({len(test_caps)})\n{test_list_md}\n"
    )
    splits_path = RESULTS_DIR / "splits.md"
    splits_path.write_text(splits_content, encoding="utf-8")
    print(f"✓ Arquivo {splits_path} gerado com sucesso.")